# Wind Turbine Power Forecasting — Experiments
**SDWPF / Baidu KDD Cup 2022 Dataset**

This notebook contains all prompt engineering experiments for LLM-based wind turbine power forecasting.

| # | Experiment | Description |
|---|-----------|-------------|
| 1 | Baseline | Original prompt, 48h, raw SCADA Wspd |
| 2 | R2L Tokenization | Right-to-left digit reversal encoding |
| 3 | 6h Ablation | Modular feature engineering ablation study |
| 4 | 48h with Reasoning | Reasoning injection to diagnose 48h degradation |
| 5 | ERA5 Oracle | Replace Wspd with ERA5 reanalysis wind speed |

**Dataset alignment:** Competition Day 1 = May 1, 2020 | SCADA = UTC+8 | ERA5 = UTC


In [ ]:
import os
import re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import google.generativeai as genai

genai.configure(api_key=os.environ.get("GEMINI_API_KEY", "YOUR_GEMINI_KEY"))
client = genai.GenerativeModel("gemini-2.5-flash")

# ── Config ────────────────────────────────────────────────────
CSV_PATH       = "wtbdata_245days.csv"
FULL_CSV_PATH  = "turb1_full.csv"
TURB_ID        = 1
BASE_DAY       = 1
FEATURE_COLS   = ["Wspd", "Wdir", "Etmp", "Patv"]
FORECAST_STEPS = 288          # 48h at 10-min resolution
COMP_START     = pd.Timestamp("2020-05-01")

print("Setup complete.")

In [ ]:

turb1 = pd.read_csv("turb1_full.csv")
turb1["Tmstamp"] = pd.to_datetime(turb1["Tmstamp"])
turb1 = turb1.sort_values("Tmstamp").reset_index(drop=True)

In [ ]:
# find comp date: 05 01 2020
# Evidence:

#   - Sliding window MSE on ERA5 Wspd_w , best match May 1 2020
#   - Visual alignment confirmed with +8h SCADA shift, -8h ERA5 shift
#   - Competition SCADA is UTC+8, ERA5 is UTC

COMP_START = pd.Timestamp("2020-05-01")
print(f"Competition Day 1 = {COMP_START.date()}")
print(f"Competition Day 245 = {(COMP_START + pd.Timedelta(days=244)).date()}")

In [ ]:

turb1_shifted = turb1.copy()
turb1_shifted["Tmstamp"] = turb1_shifted["Tmstamp"] + pd.Timedelta(hours=8)
print(f"Original start:  {turb1['Tmstamp'].min()}")
print(f"Shifted start:   {turb1_shifted['Tmstamp'].min()}")

In [ ]:
comp = pd.read_csv(CSV_PATH)

comp20 = comp[(comp["TurbID"] == TURB_ID) & (comp["Day"] <= 20)].reset_index(drop=True)
comp20["datetime"] = COMP_START + pd.to_timedelta(comp20.index * 10, unit="min")

enh_start = COMP_START
enh_end   = COMP_START + pd.Timedelta(days=20)
enh20     = turb1_shifted[
    (turb1_shifted["Tmstamp"] >= enh_start) &
    (turb1_shifted["Tmstamp"] <  enh_end)
].dropna(subset=["Wspd"]).reset_index(drop=True)

comp20["day_x"] = (comp20["datetime"] - COMP_START).dt.total_seconds() / 86400
enh20["day_x"]  = (enh20["Tmstamp"]  - COMP_START).dt.total_seconds() / 86400

# ERA5 Wspd_w needs -8h (it's already UTC, don't double-shift)
enh20["era5_day_x"] = (enh20["Tmstamp"] - pd.Timedelta(hours=8) - COMP_START).dt.total_seconds() / 86400

fig, axes = plt.subplots(3, 1, figsize=(28, 10))

axes[0].plot(comp20["day_x"],     comp20["Wspd"],  label="Competition Wspd",  color="steelblue", linewidth=0.8)
axes[0].plot(enh20["day_x"],      enh20["Wspd"],   label="Enhanced Wspd",     color="green",     linewidth=0.8)
axes[0].plot(enh20["era5_day_x"], enh20["Wspd_w"], label="ERA5 Wspd_w",       color="orange",    linewidth=0.8)
axes[0].set_ylabel("Wind speed (m/s)")
axes[0].legend()
axes[0].set_title("20-day alignment check (SCADA +8h, ERA5 UTC)")

axes[1].plot(comp20["day_x"], comp20["Etmp"], label="Competition Etmp", color="steelblue", linewidth=0.8)
axes[1].plot(enh20["day_x"],  enh20["Etmp"],  label="Enhanced Etmp",   color="green",     linewidth=0.8)
axes[1].set_ylabel("Temperature (C)")
axes[1].legend()

axes[2].plot(comp20["day_x"], comp20["Patv"], label="Competition Patv", color="steelblue", linewidth=0.8)
axes[2].plot(enh20["day_x"],  enh20["Patv"],  label="Enhanced Patv",   color="green",     linewidth=0.8)
axes[2].set_ylabel("Power (kW)")
axes[2].legend()

for ax in axes:
    ax.set_xlabel("Day")
    ax.set_xticks(range(0, 21))
    ax.grid(axis="x", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
# build era5 data in 10 min intervals
era5 = turb1[["Tmstamp", "Wspd_w"]].set_index("Tmstamp")
era5_10min = era5.resample("10min").interpolate(method="time")

comp_turb = comp[comp["TurbID"] == TURB_ID].copy().reset_index(drop=True)
comp_turb["datetime"] = COMP_START + pd.to_timedelta(comp_turb.index * 10, unit="min")
comp_turb["Wspd_era5"] = era5_10min["Wspd_w"].reindex(comp_turb["datetime"]).values

print(f"ERA5 NaNs in competition window: {comp_turb['Wspd_era5'].isna().sum()}")
print(comp_turb[["datetime", "Day", "Wspd", "Wspd_era5"]].head(5))

### Load Data

In [ ]:
comp = pd.read_csv(CSV_PATH)

turb1 = pd.read_csv(FULL_CSV_PATH)
turb1["Tmstamp"] = pd.to_datetime(turb1["Tmstamp"])

# ERA5 is UTC, SCADA is UTC+8 — shift ERA5 +8h to align with SCADA
turb1_shifted = turb1.copy()
turb1_shifted["Tmstamp"] = turb1_shifted["Tmstamp"] + pd.Timedelta(hours=8)

# build competition window with calendar datetimes
comp_turb = comp[comp["TurbID"] == TURB_ID].copy().reset_index(drop=True)
comp_turb["datetime"] = COMP_START + pd.to_timedelta(comp_turb.index * 10, unit="min")

# ERA5 resampled to 10-min, aligned to competition timestamps (no shift — original UTC matches COMP_START)
era5 = turb1[["Tmstamp", "Wspd_w"]].set_index("Tmstamp")
era5_10min = era5.resample("10min").interpolate(method="time")
comp_turb["Wspd_era5"] = era5_10min["Wspd_w"].reindex(comp_turb["datetime"]).values

print(f"Competition rows: {len(comp):,}")
print(f"Full dataset rows: {len(turb1):,}")
print(f"ERA5 NaNs in competition window: {comp_turb['Wspd_era5'].isna().sum()}")

### Shared Utilities

In [ ]:
def evaluate(forecast, ground_truth):
    gt   = np.array(ground_truth, dtype=float)
    pred = np.array(forecast,     dtype=float)
    mask = ~np.isnan(gt) & ~np.isnan(pred)
    gt, pred = gt[mask], pred[mask]
    mae  = np.mean(np.abs(gt - pred))
    rmse = np.sqrt(np.mean((gt - pred) ** 2))
    return {"MAE": round(mae, 2), "RMSE": round(rmse, 2), "Avg": round((mae + rmse) / 2, 2)}


def parse_forecast(response_text, expected=FORECAST_STEPS, r2l_decode=False):
    reasoning = re.search(r"<reasoning>(.*?)</reasoning>", response_text, re.DOTALL)
    if reasoning:
        print("\n── LLM Reasoning ──────────────────────────────")
        print(reasoning.group(1).strip())
        print("───────────────────────────────────────────────\n")

    text = re.sub(r"<reasoning>.*?</reasoning>", "", response_text, flags=re.DOTALL)
    text = re.sub(r"```[\w]*\n?", "", text)

    try:
        values = json.loads(text.strip())["forecast"]
    except:
        nums   = re.findall(r"-?\d+\.?\d*", text)
        values = [float(n) for n in nums]
        if len(values) > expected:
            values = values[-expected:]

    if r2l_decode:
        values = [decode_r2l(v, decimals=0) for v in values]

    values = [max(0, min(1500, float(v))) for v in values]
    if len(values) < expected:
        values += [values[-1]] * (expected - len(values))
    return values[:expected]


def plot_forecast(ground_truth, forecast, title, hours_total=48, save_path=None):
    hours = np.linspace(0, hours_total, len(forecast))
    gt    = np.array(ground_truth[:len(forecast)], dtype=float)
    plt.figure(figsize=(14, 5))
    plt.plot(hours, gt,       label="Ground truth", color="steelblue")
    plt.plot(hours, forecast, label="Forecast",     color="orange", linestyle="--")
    plt.xlabel("Time (hours)")
    plt.ylabel("Patv (kW)")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.show()


def get_ground_truth(base_day, steps=FORECAST_STEPS):
    days = [base_day + 14, base_day + 15]
    return comp[
        (comp["TurbID"] == TURB_ID) &
        (comp["Day"].isin(days))
    ]["Patv"].values[:steps]


print("Utilities ready.")

## Experiment 1 — Baseline (48h, Raw SCADA Wspd)
Original prompt with no modifications. 14 days of history → predict 288 steps (48h).


In [ ]:
def call_llm_baseline(base_day, feature_cols=FEATURE_COLS):
    turbine_data  = comp[comp["TurbID"] == TURB_ID].dropna(subset=["Wspd"])
    required_cols = ["Day", "Tmstamp"] + feature_cols
    end_day       = base_day + 13

    window_data = turbine_data[
        (turbine_data["Day"] >= base_day) &
        (turbine_data["Day"] <= end_day)
    ][required_cols].copy()

    data_str = window_data.to_csv(index=False, sep=",")

    prompt = f"""
Context: You are an expert wind turbine power forecasting model.
You are given 14 days of historical SCADA data for ONE turbine.
The data is sampled every 10 minutes (144 rows per day).
Columns provided: {', '.join(required_cols)}

Input Data:
{data_str}

Your task:
Predict the Active Power (Patv, in kW) for the NEXT 48 HOURS
(288 time steps at 10-minute resolution).

Instructions:
Learn the wind-speed to power relationship from the historical data.
Power is roughly proportional to wind speed cubed at low speeds.
Power saturates near rated power (~1500 kW).
Power is near zero at very low wind speeds.
Capture daily cyclic patterns.
Do NOT copy the last day.
Do NOT output negative values.
Clip values to the realistic range [0, 1500].

OUTPUT FORMAT REQUIREMENTS (CRITICAL):
Return valid JSON in the following format:
{{
  "forecast": [f1, f2, f3, ..., f288]
}}
The list MUST contain exactly 288 numbers.
No text outside the JSON. No explanations. No markdown. Only raw JSON.
"""
    response = client.generate_content(prompt)
    return response.text

In [ ]:
raw      = call_llm_baseline(BASE_DAY)
forecast = parse_forecast(raw)
gt       = get_ground_truth(BASE_DAY)
metrics  = evaluate(forecast, gt)
print(f"MAE={metrics['MAE']}  RMSE={metrics['RMSE']}  Avg={metrics['Avg']}")
plot_forecast(gt, forecast, f"Experiment 1: Baseline 48h — Turbine {TURB_ID}", save_path="exp1_baseline.png")

## Experiment 2 — R2L Tokenization
Digits of each number are reversed around the decimal point before being sent to the model.
The model is asked to output in R2L format, which is then decoded back.

**Example:** `8.4 → 4.8` | `1500 → 0051` | `12.0 → 0.21`


In [ ]:
def r2l(value, decimals=1):
    negative  = value < 0
    formatted = f"{abs(value):.{decimals}f}".replace(".", "")
    rev       = formatted[::-1]
    if decimals > 0:
        result = (rev[:-decimals] or "0") + "." + rev[-decimals:]
    else:
        result = rev
    return f"-{result}" if negative else result

def decode_r2l(encoded, decimals=0):
    try:
        neg      = str(encoded).startswith("-")
        s        = str(encoded).lstrip("-")
        parts    = s.split(".")
        int_part = parts[0]
        dec_part = parts[1].ljust(decimals, "0") if len(parts) > 1 else ""
        digits   = (int_part + dec_part)[::-1]
        if decimals > 0:
            val = float(digits[:-decimals] + "." + digits[-decimals:])
        else:
            val = float(digits)
        return -val if neg else val
    except:
        return 0.0

def encode_df_r2l(df):
    df = df.copy()
    df["Wspd"] = df["Wspd"].apply(lambda v: r2l(v, 1))
    df["Patv"] = df["Patv"].apply(lambda v: r2l(v, 0))
    return df

def call_llm_r2l(base_day):
    end_day = base_day + 15
    data = comp[
        (comp["TurbID"] == TURB_ID) &
        (comp["Day"] >= base_day) &
        (comp["Day"] <= end_day)
    ][["Day", "Tmstamp", "Wspd", "Patv"]].dropna().copy()

    encoded  = encode_df_r2l(data)
    data_str = encoded.to_csv(index=False)

    prompt = f"""Context: You are an expert wind turbine power forecasting model.
You are given 16 days of historical SCADA data for ONE turbine.
The data is sampled every 10 minutes (144 rows per day).

NUMBER ENCODING: All numeric values use Right-to-Left (R2L) digit encoding.
Digits are reversed around the decimal point.
Examples: 8.4 -> 4.8 | 12.0 -> 0.21 | 750 -> 057 | 1500 -> 0051
To decode: reverse all digits, keep decimal at same position from right.
Your forecast output must ALSO use R2L encoding.

Input Data (R2L encoded):
{data_str}

Task: Predict Active Power (Patv) for the NEXT 48 HOURS (288 timesteps, 10-min resolution).

Instructions:
- Learn the wind-speed to power relationship from the historical data
- Power is roughly proportional to wind speed cubed at low speeds
- Power saturates near rated power (R2L encoded: 0051 kW)
- Power is near zero at very low wind speeds
- Capture daily cyclic patterns
- Do NOT copy the last day
- All output values must be R2L encoded
- Clip to realistic range [0, 1500] kW before encoding

OUTPUT FORMAT (CRITICAL):
Return valid JSON only:
{{"forecast": [f1, f2, f3, ..., f288]}}
Exactly 288 R2L-encoded numbers. No text outside the JSON.
"""
    response = client.generate_content(prompt)
    return response.text

In [ ]:
raw      = call_llm_r2l(BASE_DAY)
forecast = parse_forecast(raw, r2l_decode=True)
gt       = get_ground_truth(BASE_DAY)
metrics  = evaluate(forecast, gt)
print(f"MAE={metrics['MAE']}  RMSE={metrics['RMSE']}  Avg={metrics['Avg']}")
plot_forecast(gt, forecast, f"Experiment 2: R2L Tokenization 48h — Turbine {TURB_ID}", save_path="exp2_r2l.png")

## Experiment 3 — 6h Forecast Ablation Study
Tests four prompt engineering techniques individually and in combination:
- **Stats injection** — min/max/mean/std of Wspd and Patv
- **Power curve injection** — empirical Wspd → Patv lookup table
- **Numeric rounding** — reduce token count (Wspd→1dp, Patv→int, Wdir→nearest 10°)
- **Reasoning block** — model explains its strategy before forecasting


In [ ]:
FORECAST_STEPS_6H = 36
RESULTS_PATH      = "ablation_results.csv"
FORECASTS_PATH    = "ablation_forecasts.json"

def compute_stats(df):
    return (
        f"Wspd -> min: {df['Wspd'].min():.1f}, max: {df['Wspd'].max():.1f}, "
        f"mean: {df['Wspd'].mean():.1f}, std: {df['Wspd'].std():.1f}\n"
        f"Patv -> min: {df['Patv'].min():.0f}, max: {df['Patv'].max():.0f}, "
        f"mean: {df['Patv'].mean():.0f}, std: {df['Patv'].std():.0f}"
    )

def compute_power_curve(df, bin_width=0.5):
    bins = np.arange(0, 25.5, bin_width)
    df   = df.copy()
    df["wspd_bin"] = pd.cut(df["Wspd"], bins=bins)
    curve = df.groupby("wspd_bin", observed=True)["Patv"].mean().round(0).reset_index()
    curve["wspd_mid"] = curve["wspd_bin"].apply(lambda x: round(x.mid, 1))
    curve = curve.dropna().rename(columns={"Patv": "mean_patv"})
    return "\n".join(f"  {row.wspd_mid:.1f} m/s -> {int(row.mean_patv)} kW" for _, row in curve.iterrows())

def apply_rounding(df):
    df = df.copy()
    df["Wspd"] = df["Wspd"].round(1)
    df["Patv"] = df["Patv"].round(0).astype("Int64")
    if "Wdir" in df.columns:
        df["Wdir"] = (df["Wdir"] / 10).round(0).astype("Int64") * 10
    if "Etmp" in df.columns:
        df["Etmp"] = df["Etmp"].round(0).astype("Int64")
    return df

def call_llm_6h(base_day, feature_cols, config):
    turbine_data  = comp[comp["TurbID"] == TURB_ID].dropna(subset=["Wspd"])
    end_day       = base_day + 13
    required_cols = ["Day", "Tmstamp"] + feature_cols
    window_data   = turbine_data[
        (turbine_data["Day"] >= base_day) &
        (turbine_data["Day"] <= end_day)
    ][required_cols].copy()

    stats_block = f"\nHISTORICAL STATISTICS:\n{compute_stats(window_data)}\n" if config["use_stats"] else ""
    curve_block = f"\nEMPIRICAL POWER CURVE (treat as ground truth):\n{compute_power_curve(window_data)}\n" if config["use_power_curve"] else ""
    if config["use_rounding"]:
        window_data = apply_rounding(window_data)
    data_str = window_data.to_csv(index=False, sep=",")

    reasoning_instruction = ""
    reasoning_format      = ""
    if config["use_reasoning"]:
        reasoning_instruction = "\nBefore forecasting, explain:\n1. Wind speed trend at end of data\n2. Which part of power curve applies\n3. Expected power level and why\nWrap in <reasoning> tags.\n"
        reasoning_format      = "First output <reasoning>...</reasoning>, then the JSON.\n"

    prompt = f"""Context: You are an expert wind turbine power forecasting model.
You are given 14 days of historical SCADA data for ONE turbine.
The data is sampled every 10 minutes (144 rows per day).
Columns provided: {', '.join(required_cols)}
{stats_block}{curve_block}
Input Data:
{data_str}

Your task: Predict Patv (kW) for the NEXT 6 HOURS ({FORECAST_STEPS_6H} timesteps at 10-min resolution).
{reasoning_instruction}
Instructions:
Learn the wind-speed to power relationship from the historical data.
Power proportional to Wspd^3 at low speeds. Saturates near 1500 kW. Near zero below ~3 m/s.
Capture daily cyclic patterns. Do NOT copy the last day.
Do NOT output negative values. Clip to [0, 1500].

OUTPUT FORMAT:
{reasoning_format}Return valid JSON:
{{
  "forecast": [f1, f2, ..., f{FORECAST_STEPS_6H}]
}}
Exactly {FORECAST_STEPS_6H} numbers. No markdown. Only raw JSON after any reasoning block.
"""
    response = client.generate_content(prompt)
    return response.text

def run_ablation(experiments):
    gt = comp[(comp["TurbID"] == TURB_ID) & (comp["Day"] == BASE_DAY + 14)]["Patv"].values[:FORECAST_STEPS_6H]

    existing  = pd.read_csv(RESULTS_PATH) if os.path.exists(RESULTS_PATH) else pd.DataFrame()
    done      = set(existing["experiment"].tolist()) if not existing.empty else set()
    forecasts = json.load(open(FORECASTS_PATH)) if os.path.exists(FORECASTS_PATH) else {}
    results   = existing.to_dict("records") if not existing.empty else []

    if done:
        print(f"Resuming — already done: {done}")

    for name, config in experiments.items():
        if name in done:
            print(f"Skipping: {name}")
            continue
        print(f"\nRunning: {name}")
        try:
            raw      = call_llm_6h(BASE_DAY, FEATURE_COLS, config)
            forecast = parse_forecast(raw, expected=FORECAST_STEPS_6H)
            metrics  = evaluate(forecast, gt)
            print(f"  MAE={metrics['MAE']}  RMSE={metrics['RMSE']}  Avg={metrics['Avg']}")
            forecasts[name] = forecast
            json.dump(forecasts, open(FORECASTS_PATH, "w"))
            results.append({"experiment": name, **config, **metrics})
            pd.DataFrame(results).to_csv(RESULTS_PATH, index=False)
            print(f"  Saved.")
        except Exception as e:
            print(f"  FAILED: {e}")

    return pd.DataFrame(results).sort_values("Avg")

print("Ablation functions ready.")

In [ ]:
experiments = {
    "baseline":         {"use_stats": False, "use_power_curve": False, "use_rounding": False, "use_reasoning": False},
    "stats only":       {"use_stats": True,  "use_power_curve": False, "use_rounding": False, "use_reasoning": False},
    "power curve only": {"use_stats": False, "use_power_curve": True,  "use_rounding": False, "use_reasoning": False},
    "rounding only":    {"use_stats": False, "use_power_curve": False, "use_rounding": True,  "use_reasoning": False},
    "reasoning only":   {"use_stats": False, "use_power_curve": False, "use_rounding": False, "use_reasoning": True},
    "stats + curve":    {"use_stats": True,  "use_power_curve": True,  "use_rounding": False, "use_reasoning": False},
    "curve + rounding": {"use_stats": False, "use_power_curve": True,  "use_rounding": True,  "use_reasoning": False},
    "all techniques":   {"use_stats": True,  "use_power_curve": True,  "use_rounding": True,  "use_reasoning": True},
}

results_df = run_ablation(experiments)
print("\n── Ablation Results (sorted by Avg) ──────────────")
print(results_df[["experiment", "MAE", "RMSE", "Avg"]].to_string(index=False))

## 48h Forecast with Reasoning experiment
Adds a `<reasoning>` block to the original 48h prompt to understand why accuracy degrades beyond 6h.
The model must explain its wind trend reading, power curve interpretation, and forecasting strategy before outputting numbers.


In [ ]:
def call_llm_reasoning(base_day, feature_cols=FEATURE_COLS):
    turbine_data  = comp[comp["TurbID"] == TURB_ID].dropna(subset=["Wspd"])
    required_cols = ["Day", "Tmstamp"] + feature_cols
    end_day       = base_day + 13
    window_data   = turbine_data[
        (turbine_data["Day"] >= base_day) &
        (turbine_data["Day"] <= end_day)
    ][required_cols].copy()
    data_str = window_data.to_csv(index=False, sep=",")

    prompt = f"""
Context: You are an expert wind turbine power forecasting model.
You are given 14 days of historical SCADA data for ONE turbine.
The data is sampled every 10 minutes (144 rows per day).
Columns provided: {', '.join(required_cols)}

Input Data:
{data_str}

Your task: Predict Patv (kW) for the NEXT 48 HOURS (288 timesteps at 10-min resolution).

Before forecasting, explain your reasoning inside <reasoning> tags:
1. What is the wind speed level and trend at the END of the 14-day window?
2. What daily cyclic pattern do you observe?
3. What wind-to-power relationship did you learn?
4. What is your forecasting strategy for the full 48h?
5. What are you most uncertain about and why?

Instructions:
Learn the wind-speed to power relationship from the historical data.
Power proportional to Wspd^3 at low speeds. Saturates near 1500 kW. Near zero below ~3 m/s.
Capture daily cyclic patterns. Do NOT copy the last day.
Do NOT output negative values. Clip to [0, 1500].

OUTPUT FORMAT:
<reasoning>
...
</reasoning>

{{
  "forecast": [f1, f2, ..., f288]
}}
Exactly 288 numbers. No text outside reasoning block and JSON. No markdown.
"""
    response = client.generate_content(prompt)
    return response.text

In [ ]:
raw      = call_llm_reasoning(BASE_DAY)
forecast = parse_forecast(raw)   # reasoning block printed automatically
gt       = get_ground_truth(BASE_DAY)
metrics  = evaluate(forecast, gt)
print(f"MAE={metrics['MAE']}  RMSE={metrics['RMSE']}  Avg={metrics['Avg']}")
plot_forecast(gt, forecast, f"Experiment 4: 48h with Reasoning — Turbine {TURB_ID}", save_path="exp4_reasoning.png")

## ERA5 Oracle experiment


In [ ]:
def get_window_era5(base_day, history_days=14):
    window = comp_turb[
        (comp_turb["Day"] >= base_day) &
        (comp_turb["Day"] <= base_day + history_days - 1)
    ].copy()
    window["Wspd"] = window["Wspd_era5"]
    required_cols  = ["Day", "Tmstamp", "Wspd", "Wdir", "Etmp", "Patv"]
    return window[required_cols].dropna(subset=["Wspd"])

def save_gemini_web_csv(base_day, output_path="gemini_input_era5.csv"):
    """Save token-efficient ERA5 CSV for Gemini web browser experiment."""
    window = get_window_era5(base_day)
    window = window.copy()
    window["Wspd"] = window["Wspd"].round(1)
    window["Etmp"] = window["Etmp"].round(0).astype("Int64")
    window["Patv"] = window["Patv"].round(0).astype("Int64")
    if "Wdir" in window.columns:
        window["Wdir"] = (window["Wdir"] / 10).round(0).astype("Int64") * 10
    window.to_csv(output_path, index=False)
    print(f"Saved {output_path} | Rows: {len(window)} | Days: {window['Day'].min()}–{window['Day'].max()}")
    return window

def call_llm_era5(base_day):
    turbine_data         = comp_turb.copy()
    turbine_data["Wspd"] = era5_10min["Wspd_w"].reindex(turbine_data["datetime"]).values
    turbine_data         = turbine_data.dropna(subset=["Wspd"])
    end_day              = base_day + 13
    required_cols        = ["Day", "Tmstamp", "Wspd", "Wdir", "Etmp", "Patv"]
    window_data = turbine_data[
        (turbine_data["Day"] >= base_day) &
        (turbine_data["Day"] <= end_day)
    ][required_cols].copy()
    data_str = window_data.to_csv(index=False, sep=",")

    prompt = f"""
Context: You are an expert wind turbine power forecasting model.
You are given 14 days of historical SCADA data for ONE turbine.
The data is sampled every 10 minutes (144 rows per day).
Columns provided: {', '.join(required_cols)}

Note: The Wspd column contains ERA5 reanalysis wind speed — a smoothed
regional atmospheric measurement, slightly lower than hub-level SCADA wind.

Input Data:
{data_str}

Your task: Predict Patv (kW) for the NEXT 48 HOURS (288 timesteps at 10-min resolution).

Instructions:
Learn the wind-speed to power relationship from the historical data.
Power proportional to Wspd^3 at low speeds. Saturates near 1500 kW. Near zero below ~3 m/s.
Capture daily cyclic patterns. Do NOT copy the last day.
Do NOT output negative values. Clip to [0, 1500].

OUTPUT FORMAT REQUIREMENTS (CRITICAL):
Return valid JSON in the following format:
{{
  "forecast": [f1, f2, f3, ..., f288]
}}
The list MUST contain exactly 288 numbers.
No text outside the JSON. No explanations. No markdown. Only raw JSON.
"""
    response = client.generate_content(prompt)
    return response.text

print("ERA5 functions ready.")

In [ ]:
# save CSV for Gemini web browser version
save_gemini_web_csv(BASE_DAY)

In [ ]:
# run via API
raw      = call_llm_era5(BASE_DAY)
forecast = parse_forecast(raw)
gt       = get_ground_truth(BASE_DAY)
metrics  = evaluate(forecast, gt)
print(f"MAE={metrics['MAE']}  RMSE={metrics['RMSE']}  Avg={metrics['Avg']}")
plot_forecast(gt, forecast, f"Experiment 5: ERA5 Oracle 48h — Turbine {TURB_ID}", save_path="exp5_era5.png")

## Results Summary

In [ ]:
# paste metrics from each experiment here to compare
results_summary = pd.DataFrame([
    {"experiment": "1. Baseline 48h",     "MAE": None, "RMSE": None, "Avg": None},
    {"experiment": "2. R2L Tokenization", "MAE": None, "RMSE": None, "Avg": None},
    {"experiment": "4. Reasoning 48h",    "MAE": None, "RMSE": None, "Avg": None},
    {"experiment": "5. ERA5 Oracle",      "MAE": None, "RMSE": None, "Avg": None},
])

print("Fill in metrics after running experiments:")
print(results_summary.to_string(index=False))

# ablation results loaded from file
if os.path.exists(RESULTS_PATH):
    abl = pd.read_csv(RESULTS_PATH).sort_values("Avg")
    print("\n── 6h Ablation Results ──────────────────────────")
    print(abl[["experiment", "MAE", "RMSE", "Avg"]].to_string(index=False))